# M9.2 · NIM Deployment (~20–25 min)

**All steps in this notebook:** pip, GPU, NGC key prompt, catalog NIM pull/run, optional custom Model-Free NIM from M7 SFT.

Production reference: `gsi-training/9.custom_model_deployment/README.md` (8× GPU, TP=2). Workshop: **1× GPU, TP=1**.


## 1. Prerequisites (pip + NGC + NIM)


In [1]:
import sys
from pathlib import Path

# Workshop shared helpers (parent folder: workshop-Materials/).
sys.path.insert(0, str(Path.cwd().parent))
from docker_storage import ensure_docker_storage
from notebook_env import bootstrap_notebook_env, ensure

ensure_docker_storage()  # docker images/layers on /data — avoids root disk full errors
bootstrap_notebook_env()

# M9 dependencies — installed inline so this notebook is fully self-contained.
for mod, pkg in [
    ("torch", "torch>=2.5.0,<2.7.0"),
    ("transformers", "transformers>=4.45.0,<4.55.0"),
    ("peft", "peft>=0.13.0,<0.16.0"),
    ("requests", "requests>=2.32.0"),
]:
    ensure(mod, [pkg], quiet=True)
print("Prerequisites ready.")


2026-06-10 05:27:23,355 INFO === ensure_docker_storage (log: /data/logs/docker_storage.log) ===
2026-06-10 05:27:23,356 INFO disk /: 89.1G used / 123.9G (71.9%)
2026-06-10 05:27:23,357 INFO disk /data: 228.0G used / 983.2G (705.2G free)
2026-06-10 05:27:23,357 INFO env TMPDIR=/data/cache/tmp
2026-06-10 05:27:23,358 INFO env DOCKER_TMPDIR=/data/cache/tmp
2026-06-10 05:27:23,358 INFO env PIP_CACHE_DIR=/data/cache/pip
2026-06-10 05:27:23,359 INFO env UV_CACHE_DIR=/data/cache/uv
2026-06-10 05:27:23,359 INFO env HF_HOME=/data/cache/hf
2026-06-10 05:27:23,359 INFO env XDG_CACHE_HOME=/data/cache/xdg
2026-06-10 05:27:23,360 INFO $ docker info --format {{.DockerRootDir}}
2026-06-10 05:27:23,398 INFO docker data-root (config): /data/docker
2026-06-10 05:27:23,398 INFO docker data-root (live):   /data/docker
2026-06-10 05:27:23,398 INFO $ docker info
2026-06-10 05:27:23,435 INFO $ docker info --format {{.DockerRootDir}}
2026-06-10 05:27:23,468 INFO docker ready — data-root=/data/docker, free=705.

docker storage ok: /data/docker (705.2G free on volume)
creating uv venv: /home/ubuntu/repo-content_copy/workshop-Materials/M9-nvidia_nim/.venv
notebook env: /home/ubuntu/repo-content_copy/workshop-Materials/M9-nvidia_nim/.venv (python /home/ubuntu/repo-content_copy/workshop-Materials/M9-nvidia_nim/.venv/bin/python)
installing: torch ...
installing: transformers ...
installing: peft ...
ok: requests
Prerequisites ready.


In [2]:
import torch

# Workshop runs on 1x A100 / H100 / H200 — same recipe; verify GPU here.
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    print(f"GPU: {_p.name} ({_p.total_memory / 2**30:.1f} GiB)")
else:
    print("WARNING: No CUDA GPU detected. Training notebooks will not run; Curator small-corpus paths may still work on CPU.")


GPU: NVIDIA A100 80GB PCIe (79.3 GiB)


In [3]:
import os, subprocess, time, getpass
from pathlib import Path
import requests

# Prompt for NGC API key at runtime (not stored in the notebook).
if not os.environ.get("NGC_API_KEY"):
    _key = getpass.getpass("Enter your NGC API key: ").strip()
    if _key:
        os.environ["NGC_API_KEY"] = _key
if not os.environ.get("NGC_API_KEY"):
    raise ValueError("NGC_API_KEY is required to pull and run NIM containers from nvcr.io")

LOCAL_NIM_CACHE = Path(os.environ["LOCAL_NIM_CACHE"])  # /data/cache/nim (set in cell 1)

NIM_IMAGE = "nvcr.io/nim/nvidia/nvidia-nemotron-nano-9b-v2:latest"
NIM_CONTAINER = "workshop-nim-nano"
NIM_PORT = 8080
NIM_MODEL_ID = "nvidia/nvidia-nemotron-nano-9b-v2"

print("Docker login → nvcr.io …")
subprocess.run(
    ["docker", "login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"],
    input=os.environ["NGC_API_KEY"].encode(),
    check=True,
)

def _nim_ready():
    try:
        return requests.get(f"http://localhost:{NIM_PORT}/v1/models", timeout=5).status_code == 200
    except Exception:
        return False

if _nim_ready():
    print(f"NIM already running on :{NIM_PORT}")
else:
    subprocess.run(["docker", "rm", "-f", NIM_CONTAINER], capture_output=True)
    print("Pulling NIM image (first time: several minutes) …")
    subprocess.check_call(["docker", "pull", NIM_IMAGE])
    print("Starting container …")
    subprocess.check_call([
        "docker", "run", "-d", "--name", NIM_CONTAINER,
        "--gpus", "device=0", "--shm-size", "16g",
        "-e", "NGC_API_KEY", "-e", "NIM_SERVER_PORT=8000",
        "-v", f"{LOCAL_NIM_CACHE}:/opt/nim/.cache",
        "-p", f"{NIM_PORT}:8000", NIM_IMAGE,
    ], env=os.environ.copy())
    for _i in range(40):
        if _nim_ready():
            _r = requests.get(f"http://localhost:{NIM_PORT}/v1/models", timeout=10)
            print("Models:", [m["id"] for m in _r.json().get("data", [])])
            break
        print(f"  waiting for NIM … ({_i + 1}/40)")
        time.sleep(15)
    else:
        raise RuntimeError("NIM not ready — check: docker logs workshop-nim-nano")

os.environ["LOCAL_NIM_URL"] = f"http://localhost:{NIM_PORT}/v1"
print("OpenAI endpoint:", os.environ["LOCAL_NIM_URL"])

from pathlib import Path
from docker_storage import workshop_work_dir, glob_work_paths

NB_DIR = Path.cwd().resolve()
WORK_DIR = workshop_work_dir("M9-nvidia_nim")
M7_NB = NB_DIR.parent / "M7-model_training"
M7_WORK = workshop_work_dir("M7-model_training")
_sft = glob_work_paths(M7_WORK, M7_NB, "sft_checkpoints/**/model/consolidated")
M7_SFT = _sft[-1] if _sft else (M7_NB / "work" / "sft_checkpoints" / "sft-lora-final")
MERGE_DIR = WORK_DIR / "merged_sft"
print('M7 SFT LoRA:', (M7_SFT/'adapter_config.json').exists())


Enter your NGC API key:  ········


Docker login → nvcr.io …


WARNING! Your password will be stored unencrypted in /home/ubuntu/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores



Login Succeeded
Pulling NIM image (first time: several minutes) …
latest: Pulling from nim/nvidia/nvidia-nemotron-nano-9b-v2
Digest: sha256:a2f4a5aefe7dd0ff29bfd8d7081ce4977337d1b12081361af7b6283ff9a406b2
Status: Image is up to date for nvcr.io/nim/nvidia/nvidia-nemotron-nano-9b-v2:latest
nvcr.io/nim/nvidia/nvidia-nemotron-nano-9b-v2:latest
Starting container …
0c9e3602be0354729ea0057f7ec311f08d5308032f2024552245ff32cd52c229
  waiting for NIM … (1/40)
  waiting for NIM … (2/40)
  waiting for NIM … (3/40)
  waiting for NIM … (4/40)
  waiting for NIM … (5/40)
  waiting for NIM … (6/40)
  waiting for NIM … (7/40)
  waiting for NIM … (8/40)
  waiting for NIM … (9/40)
  waiting for NIM … (10/40)
  waiting for NIM … (11/40)
  waiting for NIM … (12/40)
  waiting for NIM … (13/40)
  waiting for NIM … (14/40)
Models: ['nvidia/nvidia-nemotron-nano-9b-v2']
OpenAI endpoint: http://localhost:8080/v1
M7 SFT LoRA: False


## 2. Smoke-test catalog NIM (Nemotron Nano 9B)


In [4]:
import requests
r = requests.post(f'http://localhost:{NIM_PORT}/v1/chat/completions', json={
    'model': NIM_MODEL_ID,
    'messages': [{'role':'user','content':'In one sentence, what is AML structuring?'}],
    'max_tokens': 64,
}, timeout=120)
print(r.json()['choices'][0]['message']['content'])


Okay, the user is asking for a one-sentence definition of AML structuring. Let me start by recalling what AML stands for. AML is Anti-Money Laundering. Structuring in this context refers to the practice of breaking down large transactions into smaller ones to avoid detection. I need to make sure


## 3. Clean up

Stop and remove the NIM container(s) this notebook started, so the GPU memory and
ports are freed for the next module.

In [5]:
# Stop & remove the NIM container(s) started by this notebook.
# Listed by literal name so this works even if the custom-model cell was skipped.
import subprocess

for _name in ["workshop-nim-nano"]:
    _out = subprocess.run(["docker", "rm", "-f", _name], capture_output=True, text=True)
    if _out.returncode == 0 and _out.stdout.strip():
        print(f"removed: {_name}")
    else:
        print(f"not running / already gone: {_name}")

print("\nRemaining workshop NIM containers:")
subprocess.run(["docker", "ps", "--filter", "name=workshop-",
                "--format", "table {{.Names}}\t{{.Status}}"])

removed: workshop-nim-nano

Remaining workshop NIM containers:
NAMES     STATUS


CompletedProcess(args=['docker', 'ps', '--filter', 'name=workshop-', '--format', 'table {{.Names}}\t{{.Status}}'], returncode=0)

## 4. Deploy a custom model (Model-Free NIM) from your fine-tuned checkpoint

The catalog NIM above serves a *pre-packaged* model. To serve **your own** trained
checkpoint (e.g. the M7 SFT consolidated model), use a **Model-Free NIM**, which
loads weights from a directory you mount into the container.

Mirrors `gsi-training/9.custom_model_deployment/README.md`, scaled to **1 GPU
(`--tensor-parallel-size 1`)**. Set `CUSTOM_MODEL_DIR` to your consolidated
checkpoint directory (placeholder below — e.g. the M7 SFT output at
`../M7-model_training/work/sft_checkpoints/.../model/consolidated`).

In [ ]:
# --- Custom model: deploy your fine-tuned checkpoint behind a Model-Free NIM ---
# Mirrors gsi-training/9.custom_model_deployment/README.md (steps 1-2), scaled to
# 1 GPU (TP=1). Reuses the docker login + LOCAL_NIM_CACHE from the prereq cell.

# Workshop SFT checkpoint on /data (fallback: set CUSTOM_MODEL_DIR env var).
_sft_ckpt = glob_work_paths(M7_WORK, M7_NB, "sft_checkpoints/**/model/consolidated")
CUSTOM_MODEL_DIR = str(_sft_ckpt[-1]) if _sft_ckpt else os.environ.get("CUSTOM_MODEL_DIR", "")

CUSTOM_NIM_IMAGE     = "nvcr.io/nim/nvidia/model-free-nim:2.0.5"
CUSTOM_NIM_CONTAINER = "workshop-custom-nim"
CUSTOM_NIM_PORT      = 8088
CUSTOM_SERVED_NAME   = "aml-custom-task-nim-1"

assert CUSTOM_MODEL_DIR and Path(CUSTOM_MODEL_DIR).is_dir(), (
    f"No SFT checkpoint found on /data and CUSTOM_MODEL_DIR not set (got {CUSTOM_MODEL_DIR!r}). "
    "Run M7.3 sft.ipynb first, or export CUSTOM_MODEL_DIR to a consolidated checkpoint."
)

print("Pulling Model-Free NIM image (first time: several minutes) …")
subprocess.check_call(["docker", "pull", CUSTOM_NIM_IMAGE])

# Fresh start: remove any previous custom container on this name.
subprocess.run(["docker", "rm", "-f", CUSTOM_NIM_CONTAINER], capture_output=True)

print("Starting custom NIM (1 GPU, TP=1) …")
subprocess.check_call([
    "docker", "run", "-d",
    "--name", CUSTOM_NIM_CONTAINER,
    "--restart", "unless-stopped",
    "--runtime=nvidia",
    "--gpus", "device=0",                         # 1 GPU (README used device=0,3 for TP=2)
    "--shm-size=16g",
    "--ulimit", "memlock=-1", "--ulimit", "stack=67108864",
    "-v", f"{LOCAL_NIM_CACHE}:/opt/nim/.cache",
    "-v", f"{CUSTOM_MODEL_DIR}:{CUSTOM_MODEL_DIR}:ro",
    "-e", f"NIM_MODEL_PATH={CUSTOM_MODEL_DIR}",
    "-e", f"NIM_SERVED_MODEL_NAME={CUSTOM_SERVED_NAME}",
    "-e", "NIM_TRUST_CUSTOM_CODE=1",
    "-e", "NGC_API_KEY",
    "-p", f"{CUSTOM_NIM_PORT}:8088",
    CUSTOM_NIM_IMAGE,
    "--tensor-parallel-size", "1",                # 1 GPU (README used 2)
], env=os.environ.copy())

def _custom_ready():
    try:
        return requests.get(f"http://localhost:{CUSTOM_NIM_PORT}/v1/models", timeout=5).status_code == 200
    except Exception:
        return False

for _i in range(60):
    if _custom_ready():
        _r = requests.get(f"http://localhost:{CUSTOM_NIM_PORT}/v1/models", timeout=10)
        print("Custom NIM models:", [m["id"] for m in _r.json().get("data", [])])
        break
    print(f"  waiting for custom NIM … ({_i + 1}/60)")
    time.sleep(15)
else:
    raise RuntimeError(f"Custom NIM not ready — check: docker logs {CUSTOM_NIM_CONTAINER}")

# Smoke-test the custom endpoint.
r = requests.post(f"http://localhost:{CUSTOM_NIM_PORT}/v1/chat/completions", json={
    "model": CUSTOM_SERVED_NAME,
    "messages": [{"role": "user", "content": "In one sentence, what is AML structuring?"}],
    "max_tokens": 64,
}, timeout=120)
print("\nCustom model says:", r.json()["choices"][0]["message"]["content"])

Pulling Model-Free NIM image (first time: several minutes) …
2.0.5: Pulling from nim/nvidia/model-free-nim
Digest: sha256:ad8d7db45c63fb2980fab2b580c539e7c10dfe5d636ee3a566bc8128e59be430
Status: Image is up to date for nvcr.io/nim/nvidia/model-free-nim:2.0.5
nvcr.io/nim/nvidia/model-free-nim:2.0.5
Starting custom NIM (1 GPU, TP=1) …
651cfb4190eb9ef13c2dc377363391082ba55aedf404ee60118a5f23521d95e3
  waiting for custom NIM … (1/60)
  waiting for custom NIM … (2/60)
  waiting for custom NIM … (3/60)
  waiting for custom NIM … (4/60)
  waiting for custom NIM … (5/60)
  waiting for custom NIM … (6/60)
  waiting for custom NIM … (7/60)
  waiting for custom NIM … (8/60)
  waiting for custom NIM … (9/60)
  waiting for custom NIM … (10/60)
  waiting for custom NIM … (11/60)
  waiting for custom NIM … (12/60)
  waiting for custom NIM … (13/60)
  waiting for custom NIM … (14/60)
  waiting for custom NIM … (15/60)
  waiting for custom NIM … (16/60)
  waiting for custom NIM … (17/60)
  waiting for 

RuntimeError: Custom NIM not ready — check: docker logs workshop-custom-nim